In [ ]:
#instalacion librerias 
%pip install requests


  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
  Using cached charset_normalizer-3.4.4-cp310-cp310-win_amd64.whl (107 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
  Using cached idna-3.11-py3-none-any.whl (71 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#librerias
import requests
import ssl
import socket
from urllib.parse import urlparse

1. Utiliza Python (librería requests) para hacer una petición a un sitio web que
tenga HTTPS y otro sin HTTPS.
2. Revisa la diferencia, los encabezados y errores posibles.

In [12]:
# Ejemplo HTTP (sin cifrado)
url_http = "http://httpbin.org/get"
response_http = requests.get(url_http)
print("HTTP:", response_http.status_code)
print("URL final:", response_http.url)
print(response_http.headers)


# Ejemplo HTTPS
url_https = "https://www.shopify.com/co"
headers = {"User-Agent": "Mozilla/5.0"} # navegador para que no de 403 la peticion 
response_https = requests.get(url_https, headers=headers)
print("HTTPS:", response_https.status_code)
print("URL final:", response_https.url)
print("Encabezados HTTPS:")
print(response_https.headers)
print("-" * 50)


HTTP: 200
URL final: http://httpbin.org/get
{'Date': 'Sat, 28 Feb 2026 16:01:33 GMT', 'Content-Type': 'application/json', 'Content-Length': '307', 'Connection': 'keep-alive', 'Server': 'gunicorn/19.9.0', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Credentials': 'true'}
HTTPS: 200
URL final: https://www.shopify.com/co
Encabezados HTTPS:
{'Date': 'Sat, 28 Feb 2026 16:01:33 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'CF-Ray': '9d512427e92ce564-BOG', 'CF-Cache-Status': 'HIT', 'Age': '719', 'Cache-Control': 'max-age=900, stale-while-revalidate=86400', 'Last-Modified': 'Sat, 28 Feb 2026 15:49:34 GMT', 'Set-Cookie': '_shopify_essential_=6f6c5c99-8c6b-4d28-abd0-dce1b6a8aea5; Domain=shopify.com; Path=/; Expires=Sun, 28 Feb 2027 16:01:33 GMT; Secure; SameSite=Lax, _shopify_s=43314551-4911-4165-b80c-465954365130; Domain=shopify.com; Path=/; Expires=Sat, 28 Feb 2026 16:31:33 GMT; Secure; SameSite=Lax, _shopify_y=dae6a

3. Verifica cómo se comprueba la validez de un certificado SSL/TLS.

In [16]:
import ssl
import socket
from urllib.parse import urlparse

def verificar_certificado(url):
    try:
        parsed_url = urlparse(url)
        hostname = parsed_url.hostname
        port = 443  # puerto HTTPS

        contexto = ssl.create_default_context()

        with socket.create_connection((hostname, port), timeout=5) as sock:
            with contexto.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()

        print(f"🔐 Certificado válido de {hostname}:")
        print(" - Emitido por:", cert.get("issuer"))
        print(" - Válido desde:", cert.get("notBefore"))
        print(" - Válido hasta:", cert.get("notAfter"))
        print(" - Nombre común (CN):", cert["subject"][0][0][1])
        print("-" * 50)

    except ssl.SSLError as e:
        print(f"❌ Error SSL en {hostname}:")
        print(e)
        print("-" * 50)

    except Exception as e:
        print("❌ Error general:", e)
        print("-" * 50)


# Probar con certificado válido
verificar_certificado("https://www.shopify.com/co")

# Probar con certificado válido
verificar_certificado("https://naturalshopcol.com/?srsltid=AfmBOopK0QYApLbaGwYQkjeR7b7BtBGK_YLgjapoPc6X3UsnDSX99AjO")

# Probar con certificado vencido
verificar_certificado("https://expired.badssl.com/")

🔐 Certificado válido de www.shopify.com:
 - Emitido por: ((('countryName', 'US'),), (('organizationName', 'Google Trust Services'),), (('commonName', 'WE1'),))
 - Válido desde: Jan 10 12:18:43 2026 GMT
 - Válido hasta: Apr 10 13:18:40 2026 GMT
 - Nombre común (CN): www.shopify.com
--------------------------------------------------
🔐 Certificado válido de naturalshopcol.com:
 - Emitido por: ((('countryName', 'US'),), (('organizationName', 'Google Trust Services'),), (('commonName', 'WE1'),))
 - Válido desde: Feb  7 00:17:44 2026 GMT
 - Válido hasta: May  8 01:17:37 2026 GMT
 - Nombre común (CN): naturalshopcol.com
--------------------------------------------------
❌ Error SSL en expired.badssl.com:
[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1007)
--------------------------------------------------
